# 08. 뉴스 feature 실험

E07 `prev2` 기준선에 Home Search 월별/지역별 뉴스 aggregate를 `transaction_id` sidecar로 join해 성능 변화를 검증합니다.

- `transactions.csv`는 수정하지 않습니다.
- Home Search JSONL은 읽기 전용 입력으로만 사용합니다.
- 기본 뉴스 cutoff는 모든 거래에서 `deal_ym - 1 month`입니다.
- 비교 기준은 같은 실행의 `F10_reference_recheck`이며, 기존 E07 F10 metric도 보조 기준으로 기록합니다.

In [ ]:
from pathlib import Path
import os
import sys

current_dir = Path.cwd()
if current_dir.name == "final_project":
    PROJECT_DIR = current_dir
elif (current_dir / "final_project").exists():
    PROJECT_DIR = current_dir / "final_project"
else:
    PROJECT_DIR = Path("/Users/gwongwangjae/goorm-ai-language-course/final_project")

print("project", PROJECT_DIR)
print("python", sys.executable)
print("E08_RUN_MODE", os.environ.get("E08_RUN_MODE", "full"))
print("E08_MAX_EPOCHS", os.environ.get("E08_MAX_EPOCHS", "30"))

## 1. sidecar 생성/검증

필요하면 `scripts/build_e08_news_features.py`가 다음 산출물을 만듭니다.

- `data/external/region_month_news_signals.csv`
- `outputs/e08_news_features.csv`
- `outputs/e08_news_feature_quality_report.md`

In [ ]:
builder = PROJECT_DIR / "scripts" / "build_e08_news_features.py"
sidecar = PROJECT_DIR / "outputs" / "e08_news_features.csv"
if os.environ.get("E08_REBUILD_NEWS_FEATURES", "0") == "1" or not sidecar.exists():
    %run {builder}
else:
    print("reuse", sidecar)

## 2. 모델 실험

`F10_reference_recheck`, `F11`~`F16`을 같은 split과 residual MLP 구조로 실행합니다.

빠른 확인은 notebook 실행 전 환경변수로 조정합니다.

```bash
E08_RUN_MODE=smoke E08_MAX_EPOCHS=3
```

In [ ]:
runner = PROJECT_DIR / "scripts" / "run_e08_news_experiments.py"
%run {runner}

## 3. 산출물

- `outputs/e08_news_metrics.csv`
- `outputs/e08_news_group_metrics.csv`
- `outputs/e08_news_summary.md`

채택 여부는 `recent_holdout log_mae`, test 악화 여부, recent p99 tail 악화 여부로 판단합니다.